In [12]:
import sys
import time
import h5py
import torch
import math
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from utils import show_graph_loss, log_accuracy, run_tests

# Data loading

In [13]:
# Connect torch to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cuda


In [14]:
# Load test set
f = h5py.File('../data/experiment/sdr_wifi_test.hdf5', 'r')
X_test = f['X'][()]
y_test = f['y'][()]
f.close()

# Load train set
f = h5py.File('../data/experiment/sdr_wifi_train.hdf5', 'r')
X_train = f['X'][()]
y_train = f['y'][()]
f.close()

# Description of input variables

- `sequence_length` is the amount of IQ samples is in a single sample of data
- `nm_channels` is the amount of number each IQ samples consists of (2)
- `num_layers` is the amount of hidden layers that perform changes to get the correct prediction (chosen based on https://stats.stackexchange.com/questions/181/how-to-choose-the-number-of-hidden-layers-and-nodes-in-a-feedforward-neural-netw)
- `output_size` is the amount of frequencies that are predicted, these are equal to the input since we want to see from the full prediction which frequency most likely has the lowest interference
- `num_epochs` is the amount of training rounds
- `learning_rate` is the rate at which the weights of the hidden layers are updated to improve prediction results (cannot be too high because it might overshoot)


In [15]:
data = torch.FloatTensor(X_train).to(device)
labels = torch.FloatTensor(y_train).to(device)

print(data.shape)
print(labels.shape)

# Neural network input
sequence_length = data.shape[1]
num_channels = data.shape[2]
num_layers = 1
output_size = labels.shape[1]

learning_rate = 1e-3
num_epochs = 30
batch_size = 128

torch.Size([1120425, 64, 2])
torch.Size([1120425, 4])


# Model definition

In [16]:
USE_LOGITS = True

In [17]:
class SVMModel(nn.Module):
    def __init__(self, sequence_length, num_channels = 2, output_size = 4):
        super(SVMModel, self).__init__()

        self.fc1 = nn.Linear(2 * sequence_length, 128)
        self.fc2 = nn.Linear(128, output_size)

    def forward(self, x):
        # Step 1: flatten the data
        # Input shape: [batch_size, sequence_length, 2]
        x = x.view(x.size(0), -1)  # -> [batch_size, sequence_length * 2]

        # Step 3: use a fully connected layer to classify the signal
        x = self.fc1(x) # -> [batch_size, 128]
        x = torch.relu(x) # -> [batch_size, 128]
        x = self.fc2(x) # -> [batch_size, output_size]

        if not USE_LOGITS:
            x = torch.sigmoid(x)
        
        return x

# Initialize the model, loss function, and optimizer
model = SVMModel(
    sequence_length=sequence_length,
    num_channels=num_channels,
    output_size=output_size
).to(device)
criterion = nn.BCEWithLogitsLoss() if USE_LOGITS else nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# Training

In [18]:
dataset = TensorDataset(data, labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

avg_loss_history = []
avg_loss = 0

min_loss_history = []
max_loss_history = []

start_time = time.time()

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    min_loss = 10
    max_loss = 0

    for i, (x_batch, y_batch) in enumerate(train_loader):
        # Forward pass
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        total_loss += current_loss
        min_loss = min(min_loss, current_loss)
        max_loss = max(max_loss, current_loss)

        sys.stdout.write(f"\rEpoch [{epoch + 1}/{num_epochs}] | ({i} / {math.floor(len(data) / batch_size)}) | Loss: {loss.item():.4f} | Avg Loss last epoch: {avg_loss:.4f}")

    avg_loss = total_loss / len(train_loader)
    avg_loss_history.append(avg_loss)
    min_loss_history.append(min_loss)
    max_loss_history.append(max_loss)

    # Step the scheduler
    scheduler.step()

end_time = time.time()

training_time = end_time - start_time

Epoch [30/30] | (8753 / 8753) | Loss: 0.3121 | Avg Loss last epoch: 0.3180

# Testing and documentation

In [19]:
test_results = run_tests(
    model=model,
    criterion=criterion,
    X_test=X_test,
    y_test=y_test,
    batch_size=batch_size,
    device=device,
)

Average testing loss: 0.0025
Test accuracy: 84.5388%


In [20]:
log_accuracy(
    model='svm',
    label='SVM model (recorded data)',
    data=data,
    training_time=training_time,
    num_epochs=num_epochs,
    min_loss=min_loss,
    max_loss=max_loss,
    avg_loss=test_results['avg_loss'],
    accuracy=test_results['accuracy'],
    avg_response_time=test_results['avg_response_time'],
    sampling_rate=1
)

In [21]:
# Save model
# torch.save(model.state_dict(), '../models/svm.pth')

In [ ]:
# show_graph_loss(loss_history=avg_loss_history, num_epochs=num_epochs, label='avg', filename='../docs/results/svm/svm_avg_loss.png')
# show_graph_loss(loss_history=min_loss_history, num_epochs=num_epochs, label='min', filename='../docs/results/svm/svm_min_loss.png')
# show_graph_loss(loss_history=max_loss_history, num_epochs=num_epochs, label='max', filename='../docs/results/svm/svm_max_loss.png')

: 